# Spark Interview Prep — Medium

**Target:** Data Engineers (4–7 yrs). **Dataset:** Cabs.csv + synthetic spark.range() data.

6 topics, 13 cells each.

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    rand, col, lit, when, desc, concat, sum, count, avg,
    row_number, rank, dense_rank, broadcast
)
from pyspark.sql.window import Window

spark = (SparkSession.builder
    .appName("SparkInterviewPrep-Medium")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')

CABS_PATH = r'C:\Users\PS\Documents\Python-Exp\RawData\Cabs.csv'

cabs = (spark.read.option('header', 'true').option('inferSchema', 'true')
    .csv(CABS_PATH))
cabs.cache()
print('Cabs rows:', cabs.count())
cabs.printSchema()

## Topic 1 — Broadcast Join

Forcing the driver to broadcast a small table so every executor has a local copy,
eliminating the expensive shuffle required by SortMergeJoin.

| Aspect | Details |
|---|---|
| Default join | SortMergeJoin — full shuffle of both sides |
| Optimised join | BroadcastHashJoin — small table serialised to all executors |
| Key config | spark.sql.autoBroadcastJoinThreshold (default 10 MB) |
| Physical node | BroadcastExchange → BroadcastHashJoin |

### Business Scenario

A ride-hailing platform needs to enrich 500 000 daily trip records with cab metadata
(licence type, vehicle year) stored in a reference table of a few thousand rows.
Running this join every hour on a shared cluster, the default SortMergeJoin causes
a full shuffle of the large trips table, adding ~40 s of exchange time per run.
Switching to BroadcastHashJoin cuts the exchange entirely and reduces end-to-end
latency by 60–70 %.

In [ ]:
# BAD: regular inner join — Spark chooses SortMergeJoin (full shuffle on both sides)
orders = (spark.range(500_000)
    .withColumn('cab_number', (col('id') % 2000).cast('string'))
    .withColumn('fare', (rand() * 40 + 5).cast('double')))

cabs_ref = cabs.select('CabNumber', 'LicenseType', 'VehicleYear')

t0 = time.time()
bad_join = orders.join(cabs_ref, orders.cab_number == cabs_ref.CabNumber, 'inner')
bad_count = bad_join.count()
print(f'BAD join rows: {bad_count}  time: {time.time()-t0:.2f}s')
print('--- BAD plan ---')
bad_join.explain()

### Interview Questions

1. What is `spark.sql.autoBroadcastJoinThreshold` and what is its default value?
2. How does Spark decide between BroadcastHashJoin and SortMergeJoin automatically?
3. What happens when you broadcast a table that is too large to fit in executor memory?
4. How do you explicitly force a broadcast hint in Spark SQL and in the DataFrame API?
5. Why is BroadcastHashJoin generally faster than SortMergeJoin for small-large joins?
6. In which situations would you prefer SortMergeJoin over BroadcastHashJoin?
7. How does AQE dynamically switch a SortMergeJoin to BroadcastHashJoin at runtime?
8. How can you verify that BroadcastExchange actually appears in the physical plan?

### Logical Plan Discussion

The logical plan contains a `Join` node with a condition and join type.
The optimiser evaluates table sizes: if the smaller side is below the broadcast
threshold, it wraps it in a `ResolvedHint(broadcast)` node.
With AQE on, even if the threshold is missed at planning time, runtime statistics
collected after the first shuffle stage can trigger a late switch to broadcast.

### Physical Plan Key Nodes

- `BroadcastExchange HashedRelationBroadcastMode` — serialises small table to driver, distributes to all executors
- `BroadcastHashJoin` — probe-side scans; hash lookup against broadcast relation
- `SortMergeJoin` (bad path) — requires `Exchange hashpartitioning` on both sides before merge
- `Exchange hashpartitioning` — the shuffle node to watch for in a bad join plan

### Spark UI Guide

**SQL tab → DAG:** Look for `BroadcastExchange` (good) vs `Exchange hashpartitioning` on both sides (bad).

**Stages tab:** A broadcast join produces 1 stage for the large table scan.
A SortMergeJoin produces at least 3 stages (scan large, scan small, shuffle+merge).

**Event timeline:** `broadcastTimeout` errors show as task failures — increase
`spark.sql.broadcastTimeout` (default 300 s) for very large broadcasts.

**Metrics:** Check `shuffle bytes written` — it should be 0 for the small side in a broadcast join.

In [ ]:
# Benchmark: bad (SortMergeJoin) vs good (BroadcastHashJoin)
import time

orders2 = (spark.range(500_000)
    .withColumn('cab_number', (col('id') % 2000).cast('string'))
    .withColumn('fare', (rand() * 40 + 5).cast('double')))
cabs_ref2 = cabs.select('CabNumber', 'LicenseType', 'VehicleYear')

# Disable AQE so the comparison is fair
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')  # force SMJ
t0 = time.time()
orders2.join(cabs_ref2, orders2.cab_number == cabs_ref2.CabNumber).count()
smj_time = time.time() - t0

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))
t0 = time.time()
orders2.join(broadcast(cabs_ref2), orders2.cab_number == cabs_ref2.CabNumber).count()
bhj_time = time.time() - t0

spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))
print(f'SortMergeJoin : {smj_time:.2f}s')
print(f'BroadcastHashJoin: {bhj_time:.2f}s')
print(f'Speedup          : {smj_time/bhj_time:.1f}x')

### Typical Benchmark Results (local 4-core, 500 K rows)

| Join Type | Time |
|---|---|
| SortMergeJoin (shuffle both sides) | ~6–9 s |
| BroadcastHashJoin (no shuffle) | ~1–2 s |
| Speedup | ~4–6× |

Savings increase on a real cluster where network shuffle cost dominates.

### Best Practices

- Always use `broadcast()` hint explicitly for reference/dimension tables under ~50 MB to guarantee BHJ.
- Enable AQE so Spark can dynamically switch to BHJ even when static estimates are wrong.
- Monitor `broadcastTimeout`; increase it if the small table serialisation takes >300 s on large clusters.
- Avoid broadcasting tables that exceed ~200 MB — driver OOM and task failure risk outweigh shuffle cost.
- Cache the small table before the join if it is reused across multiple queries in the same session.

In [ ]:
# Full annotated solution: Broadcast Join

# 1. Build the large fact table (synthetic)
orders_full = (spark.range(500_000)
    .withColumn('cab_number', (col('id') % 2000).cast('string'))
    .withColumn('fare',       (rand() * 40 + 5).cast('double'))
    .withColumn('trip_date',  (col('id') % 365).cast('int')))

# 2. Prepare the small dimension table from Cabs.csv
cabs_dim = cabs.select('CabNumber', 'LicenseType', 'VehicleYear').distinct()

# 3. Explicit broadcast hint — guarantees BroadcastHashJoin regardless of threshold
enriched = (orders_full
    .join(broadcast(cabs_dim),
          orders_full.cab_number == cabs_dim.CabNumber,
          'left')
    .select('id', 'fare', 'trip_date', 'LicenseType', 'VehicleYear'))

# 4. Verify plan — must show BroadcastHashJoin, NOT SortMergeJoin
enriched.explain('formatted')

# 5. Action
print('Enriched rows:', enriched.count())

### Solution Walkthrough

1. The `broadcast()` hint wraps `cabs_dim` in a `ResolvedHint(broadcast)` logical node.
2. The optimiser emits a `BroadcastExchange` physical node that serialises the small table
   and ships it to every executor via the driver.
3. Each executor task on the large `orders_full` partitions probes the local hash map —
   no network shuffle is required for the large side.
4. `explain('formatted')` should show `BroadcastHashJoin` and `BroadcastExchange`.
5. Using `left` join ensures unmatched orders are preserved (no silent row loss).

### Practice Exercises

1. Set `spark.sql.autoBroadcastJoinThreshold = -1` and observe that `explain()` switches
   back to SortMergeJoin even with the hint applied. Research why hints override the threshold.
2. Increase the synthetic orders table to 5 M rows, run both joins, and graph the time difference.
3. Try broadcasting the large table (500 K rows) intentionally to observe the driver OOM or
   timeout behaviour; then fix it by restoring the threshold.

## Topic 2 — Shuffle Optimization

Controlling the number of shuffle (exchange) partitions to avoid thousands of tiny
tasks when data volume is small, and avoiding spill when data volume is large.

| Aspect | Details |
|---|---|
| Default partitions | 200 (spark.sql.shuffle.partitions) |
| Problem | 200 nearly-empty tasks for small aggregations |
| AQE fix | coalescePartitions merges small post-shuffle partitions |
| Physical node | Exchange hashpartitioning(N) |

### Business Scenario

An analytics pipeline runs hourly aggregations on 500 K transaction rows grouped
by 10 business categories. With the default 200 shuffle partitions, Spark spawns
200 reduce tasks of which ~190 process almost nothing, flooding the task scheduler
and inflating stage duration. Reducing shuffle partitions to 8 (matching the category
count) or relying on AQE coalescing cuts stage time by 50–70 % and reduces
task-manager overhead on the driver.

In [ ]:
# BAD: default 200 shuffle partitions on small grouped data
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '200')  # BAD: way too many for 10 groups

txn = (spark.range(500_000)
    .withColumn('category', (col('id') % 10).cast('string'))
    .withColumn('amount',   (rand() * 100).cast('double')))

t0 = time.time()
bad_agg = txn.groupBy('category').agg(sum('amount').alias('total'))
bad_agg.show()
print(f'BAD (200 partitions): {time.time()-t0:.2f}s')
bad_agg.explain()

### Interview Questions

1. What is `spark.sql.shuffle.partitions` and why is the default of 200 often wrong?
2. How does AQE's coalescePartitions feature decide how many partitions to merge?
3. What is Tungsten sort and how does it improve shuffle performance?
4. When does Spark spill shuffle data to disk, and how do you detect it in the Spark UI?
5. How do you choose an appropriate shuffle partition count for a given dataset?
6. What is the difference between `repartition()` and `coalesce()` in the context of shuffles?
7. How does `spark.sql.adaptive.coalescePartitions.minPartitionSize` affect AQE behaviour?
8. Why can too few shuffle partitions be just as bad as too many?

### Logical Plan Discussion

The logical plan shows an `Aggregate` node above a `LocalRelation` or `LogicalRDD`.
During physical planning Spark inserts an `Exchange hashpartitioning(N)` node before
the final `HashAggregate` to co-locate rows with the same key.
The value of N comes from `spark.sql.shuffle.partitions` at planning time, unless
AQE is enabled — in which case the initial plan uses N=200 but the runtime plan
coalesces down to as few partitions as `minPartitionNum` allows.

### Physical Plan Key Nodes

- `Exchange hashpartitioning(category, N)` — the shuffle node; N is the partition count
- `HashAggregate (partial)` — map-side combiner before the shuffle
- `HashAggregate (final)` — reduce-side aggregation after the shuffle
- `AQEShuffleRead coalesced` — inserted by AQE to merge small post-shuffle partitions
- `AdaptiveSparkPlan isFinalPlan=true` — wrapper added by AQE around the whole plan

### Spark UI Guide

**Stages tab → Tasks:** With 200 partitions you will see 200 tasks; most have
0 bytes input and complete in <1 ms — a clear sign of partition over-splitting.

**SQL tab → Metrics:** Check `shuffle bytes written` and `shuffle records written`.
Divide records by partition count to see average records per partition.

**Stage detail → Summary Metrics:** Look for `Spill (memory)` and `Spill (disk)`
columns — non-zero values mean shuffle data overflowed to disk (increase memory or
reduce partition size).

**With AQE:** The SQL DAG shows `AQEShuffleRead` replacing the raw `Exchange` node.

In [ ]:
# Benchmark: 200 partitions vs 8 partitions vs AQE coalesce
txn2 = (spark.range(500_000)
    .withColumn('category', (col('id') % 10).cast('string'))
    .withColumn('amount',   (rand() * 100).cast('double')))

spark.conf.set('spark.sql.adaptive.enabled', 'false')

spark.conf.set('spark.sql.shuffle.partitions', '200')
t0 = time.time()
txn2.groupBy('category').agg(sum('amount')).collect()
t200 = time.time() - t0

spark.conf.set('spark.sql.shuffle.partitions', '8')
t0 = time.time()
txn2.groupBy('category').agg(sum('amount')).collect()
t8 = time.time() - t0

spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '200')
t0 = time.time()
txn2.groupBy('category').agg(sum('amount')).collect()
tAQE = time.time() - t0

print(f'200 partitions (AQE off): {t200:.2f}s')
print(f'  8 partitions (AQE off): {t8:.2f}s')
print(f'200 partitions (AQE on) : {tAQE:.2f}s')

### Typical Benchmark Results (local 4-core, 500 K rows, 10 groups)

| Config | Time |
|---|---|
| 200 shuffle partitions, AQE off | ~3–5 s |
| 8 shuffle partitions, AQE off | ~1–2 s |
| 200 shuffle partitions, AQE on | ~1–2 s |

AQE coalesces the 200 empty partitions into ~8 at runtime, matching the manual tuning.

### Best Practices

- Enable AQE (`spark.sql.adaptive.enabled=true`) so Spark auto-coalesces empty post-shuffle partitions.
- For known small aggregations, set `spark.sql.shuffle.partitions` explicitly per query using `spark.conf.set()`.
- Target ~128 MB per shuffle partition as a starting heuristic; adjust based on Spark UI spill metrics.
- Use `repartition(N, col)` before a heavy join to pre-shuffle on the join key, then persist the result.
- Never rely solely on the default 200 for production pipelines — profile with the Spark UI first.

In [ ]:
# Full annotated solution: Shuffle Optimization

# 1. Re-enable AQE (already on from global config, but be explicit)
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '200')  # let AQE handle it

# 2. Build dataset
txn3 = (spark.range(500_000)
    .withColumn('category', (col('id') % 10).cast('string'))
    .withColumn('amount',   (rand() * 100).cast('double'))
    .withColumn('region',   (col('id') % 5).cast('string')))

# 3. Multi-level aggregation — AQE coalesces each Exchange independently
result = (txn3
    .groupBy('category', 'region')
    .agg(
        sum('amount').alias('total_amount'),
        count('id').alias('trip_count'),
        avg('amount').alias('avg_amount')
    )
    .orderBy(desc('total_amount')))

# 4. Verify plan shows AdaptiveSparkPlan and AQEShuffleRead
result.explain('formatted')
result.show()

### Solution Walkthrough

1. AQE is enabled so the initial plan starts with 200 shuffle partitions.
2. After the first shuffle stage completes, AQE reads actual partition sizes and
   merges adjacent small partitions into fewer, larger ones via `AQEShuffleRead`.
3. For 10 categories × 5 regions = 50 distinct keys, AQE coalesces down close to 50.
4. `explain('formatted')` shows `AdaptiveSparkPlan isFinalPlan=true` at the top,
   with `AQEShuffleRead` nodes wrapping the original `Exchange hashpartitioning` nodes.
5. This approach requires zero manual tuning and adapts automatically as data volume grows.

### Practice Exercises

1. Run the same aggregation with `spark.sql.adaptive.coalescePartitions.enabled=false`
   and observe the difference in task count and stage duration.
2. Increase the dataset to 5 M rows with 200 categories and find the shuffle partition
   count where performance peaks (try 50, 100, 200, 400).
3. Deliberately trigger shuffle spill by setting `spark.executor.memory=256m` in local
   mode and running a large aggregation; identify the spill metrics in the Spark UI.

## Topic 3 — Window Functions

Using window functions to rank, number, and aggregate over ordered partitions
without collapsing rows — a common pattern in leaderboards, sessionisation, and
slowly-changing-dimension logic.

| Aspect | Details |
|---|---|
| Functions | row_number, rank, dense_rank, lag, lead, sum (cumulative) |
| Key node | WindowExec with PARTITION BY + ORDER BY |
| Pitfall | Unbounded window on a huge partition → OOM |
| Alternative | groupBy + join (more shuffles, less expressive) |

### Business Scenario

The licensing authority needs a report showing, for each licence type, how cabs
rank by vehicle recency (newest first). The same report also flags the oldest cab
per licence type for renewal outreach. A naive approach uses groupBy to get the
max year, then joins back to the original table — two shuffles and complex code.
A window function delivers the same result in one pass with cleaner, auditable SQL.

In [ ]:
# BAD: simulate ranking with groupBy + join (2 shuffles, verbose)
max_year = cabs.groupBy('LicenseType').agg(
    avg('VehicleYear').alias('avg_year'),
    count('*').alias('cnt')
)
# Extra join to attach the group metric back to each row
bad_window = cabs.join(max_year, 'LicenseType', 'left')
# We still cannot produce per-row rank without another groupBy+join
t0 = time.time()
bad_window.count()
print(f'BAD (groupBy+join): {time.time()-t0:.2f}s')
bad_window.explain()

### Interview Questions

1. What is the difference between `row_number()`, `rank()`, and `dense_rank()`?
2. How does Spark physically execute a window function — what nodes appear in the plan?
3. When can a window function cause an OOM error, and how do you prevent it?
4. How do `lag()` and `lead()` differ from aggregation window functions?
5. Can you use window functions without `PARTITION BY`? What are the implications?
6. How does Spark handle ties differently in `rank()` vs `dense_rank()`?
7. What is the default window frame for aggregation functions like `sum()` in a window?
8. When should you use a window function instead of a `groupBy` aggregation?

### Logical Plan Discussion

The logical plan shows a `Window` node containing the window specification
(`partitionSpec`, `orderSpec`, `frameSpec`) and the list of window expressions.
Above it sits a `Project` to select the desired output columns.
Spark rewrites each window function call into a `WindowExpression` node during
analysis, grouping expressions that share the same window spec to minimise the
number of WindowExec stages inserted during physical planning.

### Physical Plan Key Nodes

- `WindowExec` — executes the window function over sorted, partitioned data
- `Exchange hashpartitioning(LicenseType)` — shuffles rows to co-locate each partition
- `Sort [LicenseType ASC, VehicleYear DESC]` — sorts each partition for ORDER BY
- `Window [row_number() windowspecdefinition(...)]` — the actual function evaluation
- Multiple `WindowExec` nodes stack when window specs differ (each needs its own sort)

### Spark UI Guide

**SQL tab → DAG:** Look for `Window` operator nodes. Each distinct window spec
produces its own `WindowExec` with a preceding `Sort`.

**Stage metrics:** Window functions do not reduce row count, so `output rows` equals
`input rows` — unlike groupBy which reduces rows.

**Memory:** Check `peak execution memory` per task. A single window partition
holding millions of rows can spike memory — watch for spill or OOM.

**Skew risk:** If one `LicenseType` has 90 % of rows, that partition's task will
run far longer. Check `Max Task Duration` vs `Median` in the stage detail.

In [ ]:
# Benchmark: groupBy+join vs window function
t0 = time.time()
max_yr = cabs.groupBy('LicenseType').agg(avg('VehicleYear').alias('avg_year'))
bad = cabs.join(max_yr, 'LicenseType', 'left')
bad.count()
t_bad = time.time() - t0

w = Window.partitionBy('LicenseType').orderBy(desc('VehicleYear'))
t0 = time.time()
good = (cabs
    .withColumn('rn',   row_number().over(w))
    .withColumn('rnk',  rank().over(w))
    .withColumn('drnk', dense_rank().over(w)))
good.count()
t_good = time.time() - t0

print(f'groupBy+join  : {t_bad:.2f}s')
print(f'Window function: {t_good:.2f}s')

### Typical Benchmark Results (Cabs.csv, ~few thousand rows)

| Approach | Time |
|---|---|
| groupBy + join | ~2–4 s (2 shuffles) |
| Window function | ~1–2 s (1 shuffle + sort) |

The window approach also returns a richer result (per-row rank) in a single pass.

### Best Practices

- Always specify `PARTITION BY` to limit each window partition size; avoid global windows on huge tables.
- Combine `row_number()`, `rank()`, and `dense_rank()` in one `withColumn` chain when specs match — Spark merges them into a single `WindowExec`.
- Filter early (before the window) to reduce partition sizes and avoid OOM.
- Use `rank()` when you want gaps after ties; `dense_rank()` when you want consecutive numbers.
- For top-N per group, prefer `row_number() = 1` over a join-based approach for cleaner, faster code.

In [ ]:
# Full annotated solution: Window Functions on Cabs data

# 1. Define the window: partition by licence type, newest vehicles first
w_spec = (Window
    .partitionBy('LicenseType')
    .orderBy(desc('VehicleYear')))

# 2. Apply three ranking functions in one pass (same spec = single WindowExec)
ranked = (cabs
    .filter(col('Active') == 'YES')   # filter early to shrink partitions
    .withColumn('row_num',   row_number().over(w_spec))
    .withColumn('rnk',       rank().over(w_spec))
    .withColumn('dense_rnk', dense_rank().over(w_spec)))

# 3. Top-1 per licence type (newest active cab per licence class)
top1 = ranked.filter(col('row_num') == 1)

# 4. Show results
top1.select('LicenseType', 'CabNumber', 'Name', 'VehicleYear',
            'row_num', 'rnk', 'dense_rnk').show(truncate=False)

# 5. Verify plan — look for WindowExec
ranked.explain()

### Solution Walkthrough

1. `filter(col('Active') == 'YES')` runs before the window, reducing input rows.
2. All three ranking functions share `w_spec`, so Spark emits a single `WindowExec`
   preceded by one `Exchange hashpartitioning(LicenseType)` and one `Sort`.
3. `row_number()` gives unique sequential numbers; `rank()` skips after ties;
   `dense_rank()` never skips — e.g. ranks 1,2,2,3 vs 1,2,2,4.
4. Filtering on `row_num == 1` after the window gives the top cab per group cleanly.
5. The entire operation is a single stage after the shuffle (no second join needed).

### Practice Exercises

1. Add a cumulative count of cabs per `LicenseType` ordered by `VehicleYear` using
   `count('*').over(w_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow))`.
2. Use `lag('VehicleYear', 1).over(w_spec)` to compute the year gap between consecutive
   cabs within each licence type.
3. Rank cabs by `VehicleYear` within each `VehicleType` group and return only those
   ranked in the bottom 2 (oldest vehicles) for a fleet-renewal report.

## Topic 4 — Bucketing

Pre-partitioning data on disk by a join/group key so that shuffle and sort can be
skipped entirely for matching queries — a write-once, read-many optimisation.

| Aspect | Details |
|---|---|
| API | df.write.bucketBy(N, col).sortBy(col).saveAsTable() |
| Key config | spark.sql.sources.bucketing.enabled |
| Physical node | FileScan with BucketedScan=true |
| When useful | Repeated joins / aggregations on the same key in production |

### Business Scenario

A data warehouse runs dozens of daily jobs that join a 1 M-row orders table to
a 1 K-row customers table on `customer_id`. Every run triggers an `Exchange`
hashpartitioning shuffle of the large orders table, consuming significant I/O and
network bandwidth. Writing the orders table as a bucketed Parquet table (16 buckets
on `customer_id`) means the shuffle is done once at write time; all subsequent
joins on the same key skip the exchange entirely.

In [ ]:
# BAD: write and read as plain Parquet — join always shuffles
import tempfile, os
tmp = tempfile.mkdtemp().replace('\\', '/')
plain_path = os.path.join(tmp, 'orders_plain')

orders_big = (spark.range(1_000_000)
    .withColumn('customer_id', (col('id') % 1000).cast('int'))
    .withColumn('amount',      (rand() * 200).cast('double')))

# Write plain (no bucketing)
orders_big.write.mode('overwrite').parquet(plain_path)
orders_plain = spark.read.parquet(plain_path)

customers = (spark.range(1_000)
    .withColumn('customer_id', col('id').cast('int'))
    .withColumn('name',        concat(lit('Cust_'), col('id'))))

t0 = time.time()
# BAD: triggers Exchange hashpartitioning on both sides
bad_result = orders_plain.join(customers, 'customer_id')
bad_result.count()
print(f'BAD (no bucketing): {time.time()-t0:.2f}s')

### Interview Questions

1. How does bucketing eliminate the shuffle in a join, and what conditions must both tables meet?
2. What is the significance of bucket count alignment between the two joined tables?
3. How does bucket pruning work, and when does Spark apply it?
4. Why must bucketed tables be written via `saveAsTable()` rather than `write.parquet()`?
5. What are the downsides of bucketing, and when is it counter-productive?
6. How does `sortBy()` combined with `bucketBy()` enable sort-merge without shuffle?
7. What happens to bucketed reads when `spark.sql.sources.bucketing.enabled=false`?
8. How do you inspect whether Spark is actually using a bucketed scan in the physical plan?

### Logical Plan Discussion

The logical plan for a bucketed join looks identical to a plain join — both show
a `Join` node with an equi-condition. The difference appears only during physical
planning: when Spark detects that both sides are bucketed on the join key with the
same bucket count, it emits a `SortMergeJoin` with `BucketedScan=true` on both
sides and omits the `Exchange` nodes that would otherwise precede the sort.

### Physical Plan Key Nodes

- `FileScan parquet ... BucketedScan=true` — confirms Spark is reading pre-bucketed files
- `SortMergeJoin` (without preceding `Exchange`) — shuffle-free merge join
- `Sort [customer_id ASC]` — may still appear if `sortBy()` was not used at write time
- `Exchange hashpartitioning` — absent in a correctly bucketed join (this is the win)
- `LocalTableScan` — if reading from an in-memory Hive/Spark catalog table

### Spark UI Guide

**SQL tab → DAG:** A bucketed join has no `Exchange` node before the merge —
confirm by searching for `Exchange` in the plan text; it should be absent.

**Stages tab:** A plain join produces 3+ stages (scan A, scan B, shuffle+merge).
A bucketed join produces 1 stage (scan + merge).

**Stage metrics:** `shuffle bytes written` and `shuffle read bytes` should both be 0
for the join stage in a fully bucketed scenario.

**WARN log:** If bucket counts differ, Spark logs a warning and falls back to a
regular SortMergeJoin with shuffle — watch for this in driver logs.

In [ ]:
# Benchmark: plain parquet join vs bucketed table join
import tempfile
tmp2 = tempfile.mkdtemp().replace('\\', '/')
plain2 = os.path.join(tmp2, 'orders_plain2')
bucket_tbl = 'orders_bucketed_demo'

orders_b = (spark.range(1_000_000)
    .withColumn('customer_id', (col('id') % 1000).cast('int'))
    .withColumn('amount',      (rand() * 200).cast('double')))
custs_b = (spark.range(1_000)
    .withColumn('customer_id', col('id').cast('int'))
    .withColumn('name',        concat(lit('C_'), col('id'))))

orders_b.write.mode('overwrite').parquet(plain2)
orders_b.write.mode('overwrite').bucketBy(16, 'customer_id').sortBy('customer_id')
    .saveAsTable(bucket_tbl)

ord_plain2  = spark.read.parquet(plain2)
ord_bucketed = spark.table(bucket_tbl)

t0 = time.time()
ord_plain2.join(custs_b, 'customer_id').count()
t_plain = time.time() - t0

t0 = time.time()
ord_bucketed.join(custs_b, 'customer_id').count()
t_bucket = time.time() - t0

print(f'Plain parquet join : {t_plain:.2f}s')
print(f'Bucketed table join: {t_bucket:.2f}s')

### Typical Benchmark Results (local 4-core, 1 M rows)

| Approach | Time |
|---|---|
| Plain Parquet join (shuffle) | ~8–12 s |
| Bucketed table join (no shuffle) | ~3–5 s |

The first bucketed write takes ~4–6 s but is amortised over many subsequent reads.

### Best Practices

- Always use the same bucket count on both sides of a join — mismatched counts force a fallback shuffle.
- Combine `bucketBy()` with `sortBy()` so the sort-merge join skips both the shuffle and the sort stages.
- Bucketing is best for tables accessed repeatedly in production; avoid it for one-off exploratory queries.
- Keep bucket count a power of 2 aligned to your cluster's parallelism (e.g. 16, 32, 64).
- Verify with `explain()` that `BucketedScan=true` appears; otherwise fall back to plain join tuning.

In [ ]:
# Full annotated solution: Bucketing

# 1. Write orders table bucketed on customer_id (done once)
spark.sql('DROP TABLE IF EXISTS orders_bkt_final')
orders_final = (spark.range(1_000_000)
    .withColumn('customer_id', (col('id') % 1000).cast('int'))
    .withColumn('amount',      (rand() * 200).cast('double'))
    .withColumn('status',      when(col('id') % 3 == 0, 'PAID').otherwise('PENDING')))

(orders_final.write
    .mode('overwrite')
    .bucketBy(16, 'customer_id')   # 16 buckets aligned with customers table
    .sortBy('customer_id')         # pre-sort inside each bucket
    .saveAsTable('orders_bkt_final'))

# 2. Read back and join — no Exchange should appear
custs_final = (spark.range(1_000)
    .withColumn('customer_id', col('id').cast('int'))
    .withColumn('tier',        when(col('id') % 2 == 0, 'GOLD').otherwise('SILVER')))

result_bkt = (spark.table('orders_bkt_final')
    .join(broadcast(custs_final), 'customer_id')
    .groupBy('tier', 'status')
    .agg(sum('amount').alias('total'), count('id').alias('cnt')))

result_bkt.explain('formatted')
result_bkt.show()

### Solution Walkthrough

1. `bucketBy(16, 'customer_id')` writes 16 files per partition, each file containing
   all rows where `hash(customer_id) % 16 == bucket_index`.
2. `sortBy('customer_id')` sorts rows within each bucket file on disk.
3. On read, Spark's `FileScan` recognises the bucket metadata and sets `BucketedScan=true`.
4. Because `custs_final` is small, `broadcast()` is used — the join skips both shuffle
   and sort for the orders side, leaving only a local hash probe.
5. `explain('formatted')` should show no `Exchange` nodes preceding the join.

### Practice Exercises

1. Write the same orders data with `bucketBy(32, 'customer_id')` and `bucketBy(16, 'customer_id')`
   then join them — observe the warning in the logs and the fallback to shuffle.
2. Disable bucketing globally (`spark.sql.sources.bucketing.enabled=false`) and re-run
   the bucketed join to confirm the Exchange reappears in the plan.
3. Add a second dimension table (e.g. products) bucketed on `product_id` and write a
   three-way bucketed join, verifying that neither side requires an Exchange.

## Topic 5 — Data Skew Detection

Identifying and mitigating hot keys that cause one executor task to process
orders of magnitude more data than its peers, creating stragglers.

| Aspect | Details |
|---|---|
| Symptom | One task runs 10-100× longer than median |
| Detection | groupBy(key).count().orderBy(desc) |
| AQE fix | spark.sql.adaptive.skewJoin.enabled |
| Manual fix | Salting: key + rand suffix before join |

### Business Scenario

A fraud-detection pipeline joins 500 K transactions to a merchant lookup table.
Eighty percent of test transactions share `customer_id=0` (a bot account), creating
a massive hot key. The join task for bucket 0 processes 400 K rows while all other
tasks process ~500 rows each. The straggler task keeps the entire stage alive for
minutes after every other task finishes, stalling downstream stages.

In [ ]:
# BAD: join with severe skew — customer_id=0 holds 80% of rows
skewed_orders = (spark.range(500_000)
    .withColumn('customer_id',
        when(col('id') % 5 == 0, lit(0))   # 20% go to id=0 directly
        .when(col('id') % 5 == 1, lit(0))  # another 20% -> 40% total so far
        .when(col('id') % 5 == 2, lit(0))  # now 60%
        .when(col('id') % 5 == 3, lit(0))  # now 80%
        .otherwise((col('id') % 100 + 1).cast('int')))
    .withColumn('amount', (rand() * 100).cast('double')))

merchants = (spark.range(101)
    .withColumn('customer_id', col('id').cast('int'))
    .withColumn('name', concat(lit('Merch_'), col('id'))))

t0 = time.time()
# BAD: no skew awareness — bucket 0 becomes a massive straggler
skewed_join = skewed_orders.join(merchants, 'customer_id')
skewed_join.count()
print(f'BAD (skewed join): {time.time()-t0:.2f}s')

### Interview Questions

1. How do you detect data skew programmatically before running a join?
2. What is the salting technique and how does it break up a hot key?
3. What Spark UI metrics reveal a straggler task caused by data skew?
4. How does AQE's `skewJoin` feature detect and handle skewed partitions at runtime?
5. What are `spark.sql.adaptive.skewJoin.skewedPartitionFactor` and `skewedPartitionThresholdInBytes`?
6. Can salting be applied to a broadcast join? Why or why not?
7. What is the isolate-hot-key pattern and when is it preferred over salting?
8. How do you verify that AQE skewJoin is active in the physical plan?

### Logical Plan Discussion

The logical plan is a standard equi-join — skew is invisible at the logical level.
During physical planning with AQE enabled, after the first shuffle stage Spark
collects runtime partition size statistics. If a partition's size exceeds
`skewedPartitionFactor × median_partition_size` AND exceeds `skewedPartitionThresholdInBytes`,
AQE marks it as skewed and splits it into sub-partitions, replicating the matching
build-side partitions to serve each sub-partition independently.

### Physical Plan Key Nodes

- `SortMergeJoin` with `isSkewed=true` annotation on one side — visible in AQE plan
- `AdaptiveSparkPlan isFinalPlan=true` — AQE wrapper; skew handling happens inside
- `AQEShuffleRead [skewed=true]` — the node Spark inserts to split the hot partition
- `Exchange hashpartitioning` — the shuffle that creates the skewed partitions in the first place

### Spark UI Guide

**Stages tab → Tasks:** Sort tasks by `Duration DESC`. A straggler is obvious when
the max task duration is 10-100× the median.

**Stage detail → Input Size / Records:** The skewed task will show far more input
records than its peers — compare the distribution in the task metrics table.

**SQL tab → plan text:** Search for `skewed=true` to confirm AQE is handling it.

**Event timeline:** All tasks finish quickly except one long bar — the straggler.
After AQE skewJoin splits it, you will see 2-4 shorter bars instead.

In [ ]:
# Detect skew and benchmark: naive vs AQE skewJoin

# Step 1: Detect skew distribution
print('Key distribution (top 5):')
skewed_orders.groupBy('customer_id').count().orderBy(desc('count')).show(5)

# Step 2: Bad — AQE off
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '20')
t0 = time.time()
skewed_orders.join(merchants, 'customer_id').count()
t_bad = time.time() - t0

# Step 3: Good — AQE skewJoin on
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '2')
t0 = time.time()
skewed_orders.join(merchants, 'customer_id').count()
t_good = time.time() - t0

print(f'No skew handling (AQE off): {t_bad:.2f}s')
print(f'AQE skewJoin enabled      : {t_good:.2f}s')

### Typical Benchmark Results (local 4-core, 500 K rows, 80% hot key)

| Approach | Time |
|---|---|
| No skew handling (AQE off) | ~8–15 s (straggler dominates) |
| AQE skewJoin enabled | ~3–5 s (hot partition split) |

On a cluster the gap is more dramatic — the straggler can hold a stage open for
minutes while all other executors sit idle.

### Best Practices

- Always run `groupBy(join_key).count().orderBy(desc('count'))` before writing a new join to production.
- Enable AQE skewJoin (`spark.sql.adaptive.skewJoin.enabled=true`) as a default for all production jobs.
- For extremely hot keys (>50% of data), use the isolate-hot-key pattern: handle key=X separately with broadcast, union with the rest.
- Salting (appending a random suffix 0-N to the key) works for aggregations but requires de-salting; prefer AQE for joins.
- Monitor P95/P99 task duration in the Spark UI — a large spread between median and max is the primary skew signal.

In [ ]:
# Full annotated solution: skew handling with AQE + isolate-hot-key pattern

# Re-enable AQE with aggressive skew thresholds
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '2')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes', '1048576')

# Isolate-hot-key pattern: handle customer_id=0 separately via broadcast
hot_orders  = skewed_orders.filter(col('customer_id') == 0)
cold_orders = skewed_orders.filter(col('customer_id') != 0)

# Hot path: broadcast the tiny merchant record for customer 0
merch_hot = merchants.filter(col('customer_id') == 0)
hot_result = hot_orders.join(broadcast(merch_hot), 'customer_id')

# Cold path: regular AQE-aware join (no skew here)
cold_result = cold_orders.join(merchants, 'customer_id')

# Union both paths
final_result = hot_result.union(cold_result)
print('Total rows:', final_result.count())
final_result.explain('formatted')

### Solution Walkthrough

1. The isolate-hot-key pattern separates the 80% hot-key rows from the rest.
2. The hot path uses `broadcast()` — the single merchant record for customer 0
   is trivially small, so BroadcastHashJoin completes in milliseconds.
3. The cold path contains only 20% of rows spread across ~99 distinct keys —
   AQE handles any residual imbalance automatically.
4. `union()` recombines both result sets without a shuffle (it is just a logical union).
5. `explain('formatted')` shows two separate join paths merged by a `Union` node.

### Practice Exercises

1. Implement salting for the skewed aggregation: append `(col('id') % 4).cast('string')`
   to `customer_id` as a salt, aggregate, then sum results by the original key.
2. Lower `skewedPartitionFactor` to 1 and observe whether Spark over-splits even
   non-skewed partitions; discuss the trade-off.
3. Simulate a 3-way join where two of the three tables have the same hot key and
   design a strategy to handle both skews simultaneously.

## Topic 6 — Adaptive Query Execution (AQE)

AQE re-optimises the physical plan at runtime using actual statistics collected
after each shuffle stage, enabling three key adaptations: coalescing empty shuffle
partitions, switching join strategies, and splitting skewed join partitions.

| Aspect | Details |
|---|---|
| Enabled by | spark.sql.adaptive.enabled=true (default in Spark 3.2+) |
| Plan wrapper | AdaptiveSparkPlan isFinalPlan=false → true |
| Feature 1 | coalescePartitions — merges small post-shuffle partitions |
| Feature 2 | join strategy switch — SMJ → BHJ if small side detected at runtime |
| Feature 3 | skewJoin — splits oversize partitions into sub-tasks |

### Business Scenario

A multi-stage ETL pipeline runs nightly on a cluster shared with other teams.
Data volumes fluctuate by 30–50% day-to-day depending on upstream ingestion.
Static tuning (fixed shuffle partitions, manual broadcast thresholds) causes
either under-utilisation on light nights or OOM/spill on heavy nights.
Enabling AQE allows the planner to adapt to each night's actual data shape,
reducing P99 job latency by 40% and eliminating the weekly manual re-tuning cycle.

In [ ]:
# BAD: AQE disabled — static plan, no runtime adaptation
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '200')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(5 * 1024 * 1024))

data = (spark.range(500_000)
    .withColumn('grp',    (col('id') % 15).cast('string'))
    .withColumn('amount', (rand() * 100).cast('double')))

dim = (spark.range(15)
    .withColumn('grp',   col('id').cast('string'))
    .withColumn('label', concat(lit('G'), col('id'))))

t0 = time.time()
bad_aqe = (data
    .join(dim, 'grp')
    .groupBy('label').agg(sum('amount').alias('total')))
bad_aqe.count()
print(f'AQE OFF: {time.time()-t0:.2f}s')
bad_aqe.explain()

### Interview Questions

1. What are the three main features of AQE and how does each work?
2. What does `isFinalPlan=false` mean in an AQE physical plan, and when does it become true?
3. Where does AQE get runtime statistics — what is the source of truth?
4. How does AQE's dynamic join strategy switching differ from the static broadcast threshold?
5. In what situations can AQE not help, and what static tuning is still required?
6. How do you disable only one AQE sub-feature while keeping the others active?
7. What is `spark.sql.adaptive.advisoryPartitionSizeInBytes` and how does it guide coalescing?
8. How does AQE interact with the Spark UI — what additional nodes or annotations appear?

### Logical Plan Discussion

AQE does not change the logical plan. It wraps the physical plan in an
`AdaptiveSparkPlan` node. Initially `isFinalPlan=false`: the plan contains
`QueryStage` placeholders that will be replaced with actual results after each
shuffle stage materialises. Once all stages complete, `isFinalPlan=true` and the
plan reflects the actual execution path chosen at runtime.

### Physical Plan Key Nodes

- `AdaptiveSparkPlan isFinalPlan=false` — outer wrapper; plan may still change
- `AdaptiveSparkPlan isFinalPlan=true` — final materialised plan after all stages
- `AQEShuffleRead coalesced` — coalesced partition reads replacing plain Exchange reads
- `AQEShuffleRead [skewed=true]` — split skewed partition reads
- `BroadcastHashJoin` (replacing `SortMergeJoin`) — dynamic join strategy switch
- `ShuffleQueryStage` — barrier node where AQE pauses to collect statistics

### Spark UI Guide

**SQL tab → plan text:** Look for `AdaptiveSparkPlan`. Expand it to see which
sub-features fired (coalesce, skewJoin, join switch).

**Stages tab:** With AQE coalescing, the reduce stage shows far fewer tasks than
the configured `shuffle.partitions` — confirm by comparing task count to the setting.

**SQL metrics:** `number of skewed partitions` and `number of coalesced partitions`
are reported as SQL metrics on the AQE nodes.

**Event timeline:** AQE introduces a brief pause between stages while statistics
are collected — visible as a gap in the timeline between the map and reduce stages.

In [ ]:
# Benchmark: AQE off vs AQE on (all 3 features active)
data2 = (spark.range(500_000)
    .withColumn('grp',    (col('id') % 15).cast('string'))
    .withColumn('amount', (rand() * 100).cast('double')))
dim2  = (spark.range(15)
    .withColumn('grp',   col('id').cast('string'))
    .withColumn('label', concat(lit('G'), col('id'))))

spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '200')
t0 = time.time()
(data2.join(dim2, 'grp')
    .groupBy('label').agg(sum('amount'))).count()
t_off = time.time() - t0

spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '200')
t0 = time.time()
(data2.join(dim2, 'grp')
    .groupBy('label').agg(sum('amount'))).count()
t_on = time.time() - t0

print(f'AQE OFF: {t_off:.2f}s')
print(f'AQE ON : {t_on:.2f}s')

### Typical Benchmark Results (local 4-core, 500 K rows, 15 groups)

| Config | Time |
|---|---|
| AQE off, 200 shuffle partitions | ~4–6 s |
| AQE on (coalesce + skewJoin + join switch) | ~1–3 s |

AQE coalesces the 200 shuffle partitions down to ~15, eliminating ~185 empty tasks.

### Best Practices

- Enable AQE globally in production (`spark.sql.adaptive.enabled=true`) — it is the default in Spark 3.2+.
- Set `spark.sql.adaptive.advisoryPartitionSizeInBytes` to ~128 MB to guide coalescing to a sensible target.
- Keep `spark.sql.adaptive.skewJoin.enabled=true` for join-heavy pipelines; set `skewedPartitionFactor=3` to avoid over-splitting on mildly uneven data.
- Do not disable AQE to 'simplify' debugging — instead use `explain('formatted')` which shows the final AQE plan.
- Remember AQE cannot help with data reading bottlenecks (source skew, partition file size imbalance) — those require upstream fixes.

In [ ]:
# Full annotated solution: AQE with all 3 features demonstrated

# 1. Configure AQE with all three sub-features
spark.conf.set('spark.sql.adaptive.enabled',                        'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled',     'true')
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes',   str(64 * 1024 * 1024))
spark.conf.set('spark.sql.adaptive.skewJoin.enabled',               'true')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '3')
spark.conf.set('spark.sql.shuffle.partitions',                      '200')

# 2. Data with mild skew (group 0 has 3x others)
aqe_data = (spark.range(500_000)
    .withColumn('grp', when(col('id') % 4 == 0, lit('0')).otherwise((col('id') % 10).cast('string')))
    .withColumn('val', (rand() * 50).cast('double')))
aqe_dim  = (spark.range(10)
    .withColumn('grp',   col('id').cast('string'))
    .withColumn('label', concat(lit('Cat'), col('id'))))

# 3. Join + aggregate — AQE applies all three features at runtime
aqe_result = (aqe_data
    .join(aqe_dim, 'grp')
    .groupBy('label')
    .agg(sum('val').alias('total'), count('id').alias('cnt')))

# 4. Show final plan — look for AdaptiveSparkPlan, AQEShuffleRead
aqe_result.explain('formatted')
aqe_result.show()

### Solution Walkthrough

1. `spark.sql.shuffle.partitions=200` sets the initial plan to 200 reduce tasks.
2. After the join shuffle stage, AQE reads actual partition sizes. The ~185 nearly-empty
   partitions are coalesced, leaving ~10–15 meaningful partitions (`AQEShuffleRead coalesced`).
3. AQE checks whether the `aqe_dim` side (post-shuffle) is small enough for a broadcast
   join and switches to `BroadcastHashJoin` if so — visible in the final plan.
4. The mild skew on group '0' may trigger `AQEShuffleRead [skewed=true]` depending on
   whether it exceeds `skewedPartitionFactor × median`.
5. `isFinalPlan=true` in `explain()` confirms the plan was re-optimised at runtime.

### Practice Exercises

1. Disable each AQE sub-feature one at a time (`coalescePartitions.enabled=false`,
   `skewJoin.enabled=false`) and use `explain('formatted')` to observe which AQE
   nodes disappear from the plan.
2. Set `advisoryPartitionSizeInBytes` to 1 MB (very small) and observe how many
   partitions AQE produces; discuss why too-small target sizes can hurt performance.
3. Run the full 6-topic notebook end-to-end and use the Spark History Server to
   compare the stage counts, shuffle bytes, and total duration for each topic.